# NullVector Quickstart (Postgres + OpenRouter)

Process a PDF into a searchable hierarchical tree with **4 lines of code** using `NullVectorClient`.

**Prerequisites:**
- PostgreSQL reachable via `NULLVECTOR_POSTGRES_CONNINFO`
  (default: `postgresql://REDACTED_DB_CRED@localhost:5432/app`)
- `OPENROUTER_API_KEY` set in `.env` or shell environment
- Project dependencies installed (`uv sync --extra dev`)

See `.env.example` at the repo root for all supported configuration variables.

## What is NullVector?

NullVector is a **vectorless hierarchical RAG framework** — it processes PDFs and Markdown
into auditable, searchable hierarchical trees *without vector embeddings*. All retrieval is
structural (LLM-driven ranking, tree traversal) with full traceability back to source pages.

### Pipeline at a glance

```
PDF / Markdown
    │
    ▼
┌─────────────┐    VLM page transcription (parallel) + outline extraction
│  Acquisition │    → CanonicalDocumentLedger (VLM Markdown + section anchors)
└──────┬──────┘
       ▼
┌─────────────┐    LLM hierarchy synthesis (single-shot or map-reduce for large docs)
│  Tree Build  │    → HierarchyNode tree (sections with page spans + source anchors)
└──────┬──────┘
       ▼
┌─────────────┐    Create queryable retrieval units from tree nodes
│  Retrieval   │    → RetrievalCorpus (searchable text units with page refs)
│  Index       │
└──────┬──────┘
       ▼
┌─────────────┐    LLM-driven tree search + ranking over the corpus
│  Search      │    → RetrievalHit[] (ranked results with page citations)
└──────┬──────┘
       ▼
┌─────────────┐    Grounded question answering with evidence
│  QA          │    → Answer with page-level citations
└─────────────┘
```

### Key concepts

| Concept | What it means |
|---------|---------------|
| **Acquisition** | VLM-driven page transcription with parallel processing; extracts section anchors and outlines from PDFs and Markdown |
| **Section anchor** | An HTML comment marker (`SECTION_ANCHOR`) emitted by the VLM during transcription, providing grounded heading positions that the tree builder matches to hierarchy nodes |
| **Hierarchy node** | A section in the document tree — has a title, page span, owned text, and optional source anchors for provenance |
| **Map-reduce synthesis** | For documents exceeding a page threshold (default 15), the LLM hierarchy builder uses a chunked map + merge strategy instead of a single-shot call |
| **Retrieval unit** | A searchable text chunk derived from a tree node — the atomic unit that search operates over |
| **Tree search** | Navigating the hierarchy tree to find relevant sections, then ranking their retrieval units |
| **Tree summarization** | LLM-powered bottom-up summarization of tree nodes — enriches search with semantic context |
| **Grounded QA** | Answering questions using only evidence found in the corpus — never hallucinating |

### What this notebook covers

1. **Setup** — imports, PostgreSQL connection, Groq round-robin LLM gateway
2. **Ingest** — `client.ingest()` runs acquisition → tree build → retrieval in one call
3. **New features** — parallel VLM transcription, semantic anchoring, map-reduce hierarchy synthesis
4. **Summarize** — `client.build_tree(summarize=True)` re-builds the tree with LLM summaries
5. **Describe** — `client.build_description()` generates a document-level summary
6. **Search** — `client.search()` finds relevant sections with page citations
7. **QA** — `client.ask()` answers questions with three modes (summary, focused, low-evidence)

## Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path


def banner(title: str) -> None:
    print()
    print("=" * 60)
    print(title)
    print("=" * 60)


def show_json(title: str, payload: object) -> None:
    banner(title)
    print(json.dumps(payload, indent=2, sort_keys=True, default=str))


ROOT = Path.cwd()
print(f"Repository root: {ROOT}")

### Imports

Only three NullVector imports needed for the client path. The gateway and storage
configuration are the only framework internals you touch directly.

In [ ]:
from nullvector import NullVectorClient
from nullvector.llm import GatewayAuditConfig, GatewayConfig, GatewayService
from nullvector.observability import (
    DEFAULT_OBSERVABILITY_JSONL_PATH,
    configure_default_runtime_observability,
)
from nullvector.storage import PostgresStorageConfig

try:
    import litellm

    from nullvector.llm.adapters import LiteLLMAdapter
except ImportError as exc:
    litellm = None
    LiteLLMAdapter = None
    LITELLM_IMPORT_ERROR = exc
else:
    LITELLM_IMPORT_ERROR = None

### Configuration

All paths, model names, and feature flags. Run IDs are **auto-generated by default** —
re-running creates fresh client-managed runs unless you pass explicit run IDs.

In [ ]:
COOKBOOK_RUNTIME_ROOT = ROOT / ".artifacts" / "cookbook" / "01_quickstart"
COOKBOOK_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

# Source PDF — relative to the cookbook directory
PDF_SOURCE_PATH = str(ROOT / "903000608.pdf")

# PostgreSQL connection (local-dev default; override via .env)
POSTGRES_CONNINFO = os.environ.get(
    "NULLVECTOR_POSTGRES_CONNINFO",
    "postgresql://REDACTED_DB_CRED@localhost:5432/app",
)
POSTGRES_SCHEMA = os.environ.get("NULLVECTOR_POSTGRES_SCHEMA", "public")

# LLM model (OpenRouter + Gemini Flash Lite by default)
OPENROUTER_MODEL = os.environ.get(
    "NULLVECTOR_LLM_MODEL",
    "openrouter/google/gemini-2.5-flash-lite-preview-09-2025",
)

# OpenRouter API key — REQUIRED for live VLM/LLM mode.
# Set via .env file or shell: export OPENROUTER_API_KEY=sk-or-v1-...
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")

# Feature flags
ENABLE_BRAIN = (
    os.environ.get("NULLVECTOR_NOTEBOOK_ENABLE_BRAIN", "1") != "0"
    and bool(OPENROUTER_API_KEY)
)

# Workspace root for NullVectorClient's local catalog
WORKSPACE = ROOT / ".artifacts" / "cookbook" / "01_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)


### OpenRouter Gateway

NullVector's LLM gateway owns retries internally. We use a single OpenRouter API key
with Gemini 2.5 Flash Lite as the backend model.

In [ ]:
# OpenRouter setup — single key, no rotation needed
print(f"OpenRouter model: {OPENROUTER_MODEL}")
print(f"API key: ...{OPENROUTER_API_KEY[-4:]}")

### Gateway & Client

Build the LLM gateway (live Groq round-robin when available, `None` otherwise).
When `gateway=None`, the pipeline uses outline-based fallback — no LLM/VLM calls.
When a gateway is available, VLM page transcription (parallel), LLM hierarchy synthesis
(single-shot or map-reduce), tree summarization, and LLM-backed descriptions are all enabled.

### New Pipeline Features

Three architectural features are active by default through the `general_document` preset:

| Feature | What it does | Default behavior |
|---------|-------------|-----------------|
| **Parallel VLM transcription** | Pages are rendered sequentially (PyMuPDF not thread-safe), then transcribed in parallel via `gateway.invoke_many()` | 4 concurrent pages (`vlm_max_concurrent_pages=4`) |
| **Semantic anchoring** | VLM prompt instructs models to emit `<!-- SECTION_ANCHOR: level=N, title="..." -->` before headings; these are matched to tree nodes as `source_anchors` | Always on when gateway is present |
| **Map-reduce hierarchy** | Large documents are split into overlapping chunks, analyzed in parallel, then merged into a unified hierarchy | Active for documents with 15+ pages (`hierarchy_chunking_threshold=15`) |

All three activate automatically. No code changes needed.

In [ ]:
from nullvector.presets import resolve_preset

preset = resolve_preset(None)
show_json("Default AcquisitionSettings", {
    "vlm_max_concurrent_pages": preset.acquisition_settings.vlm_max_concurrent_pages,
    "render_dpi": preset.acquisition_settings.render_dpi,
})
show_json("Default TreeSettings", {
    "hierarchy_chunk_size": preset.tree_settings.hierarchy_chunk_size,
    "hierarchy_chunk_overlap": preset.tree_settings.hierarchy_chunk_overlap,
    "hierarchy_chunking_threshold": preset.tree_settings.hierarchy_chunking_threshold,
    "max_pages_per_leaf_node": preset.tree_settings.max_pages_per_leaf_node,
})

In [ ]:
from nullvector.domain.ledger import AcquisitionSettings
from nullvector.domain.tree import TreeSettings
from nullvector.presets import DocumentPreset

custom_preset = DocumentPreset(
    name="high_concurrency",
    acquisition_settings=AcquisitionSettings(vlm_max_concurrent_pages=8),
    tree_settings=TreeSettings(
        hierarchy_chunk_size=12,
        hierarchy_chunk_overlap=3,
        hierarchy_chunking_threshold=20,
    ),
)
show_json("Custom preset example", {
    "vlm_max_concurrent_pages": custom_preset.acquisition_settings.vlm_max_concurrent_pages,
    "hierarchy_chunk_size": custom_preset.tree_settings.hierarchy_chunk_size,
    "hierarchy_chunking_threshold": custom_preset.tree_settings.hierarchy_chunking_threshold,
})
print("\nUsage: client.ingest(PDF_SOURCE_PATH, preset=custom_preset)")

In [ ]:
pg_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO, schema=POSTGRES_SCHEMA)
OBSERVABILITY_JSONL_PATH = os.environ.get(
    "NULLVECTOR_OBSERVABILITY_JSONL_PATH",
    DEFAULT_OBSERVABILITY_JSONL_PATH,
)
runtime_logger = configure_default_runtime_observability(jsonl_path=OBSERVABILITY_JSONL_PATH)


def build_demo_gateway() -> tuple[GatewayService | None, str]:
    """Build an OpenRouter gateway or return None for deterministic-only mode."""
    audit_root = COOKBOOK_RUNTIME_ROOT / "gateway_audit"
    audit_root.mkdir(parents=True, exist_ok=True)
    if not ENABLE_BRAIN:
        return None, "disabled"
    if LiteLLMAdapter is not None and litellm is not None:
        def openrouter_completion(**kwargs):
            kwargs["api_key"] = OPENROUTER_API_KEY
            # LiteLLM strips response_format for OpenRouter models.
            # Move it into extra_body so it reaches the API intact.
            rf = kwargs.pop("response_format", None)
            if rf is not None:
                extra = kwargs.setdefault("extra_body", {})
                extra["response_format"] = rf
            return litellm.completion(**kwargs)

        return (
            GatewayService(
                GatewayConfig(
                    default_model=OPENROUTER_MODEL,
                    audit=GatewayAuditConfig(persist_root=str(audit_root)),
                ),
                provider_adapter=LiteLLMAdapter(completion_fn=openrouter_completion),
                logger=runtime_logger,
            ),
            "litellm-openrouter-gemini",
        )
    if LITELLM_IMPORT_ERROR is not None:
        print(f"LiteLLM unavailable: {LITELLM_IMPORT_ERROR}")
    return None, "none (deterministic-only)"


gateway, gateway_mode = build_demo_gateway()

client = NullVectorClient(
    WORKSPACE,
    storage=pg_config,
    gateway=gateway,
    logger=runtime_logger,
)

show_json("Client ready", {
    "workspace": str(WORKSPACE),
    "gateway_mode": gateway_mode,
    "postgres_schema": POSTGRES_SCHEMA,
    "model": OPENROUTER_MODEL,
})

## Ingest — One Call Does Everything

`client.ingest()` runs the full pipeline in one call:
1. **Acquisition** — VLM page transcription (parallel, 4 pages at a time by default) + section anchor extraction + outline detection
2. **Tree Build** — LLM hierarchy synthesis (map-reduce for 15+ page docs) with source anchor matching
3. **Retrieval Index** — searchable corpus from tree nodes

No manual service construction, no manifest refs, no run ID wiring.

In [ ]:
result = client.ingest(PDF_SOURCE_PATH)

show_json("Ingest complete", {
    "document_id": result.document_id,
    "acquisition_manifest": result.acquisition_manifest_path,
    "tree_manifest": result.tree_manifest_path,
    "retrieval_manifest": result.retrieval_manifest_path,
})

### Inspecting Pipeline Artifacts

After ingest, we can inspect the new artifacts: **section anchors** embedded in the VLM
Markdown, and **source anchors** matched to hierarchy nodes. These provide spatial
provenance — each hierarchy node knows where on the source page its heading was found.

In [ ]:
from nullvector._client_utils import load_model_artifact
from nullvector.domain.ledger import AcquisitionRunManifest, CanonicalDocumentLedger

acq_manifest = load_model_artifact(
    AcquisitionRunManifest,
    result.acquisition_manifest_path,
    storage=pg_config,
)
ledger = load_model_artifact(
    CanonicalDocumentLedger,
    acq_manifest.ledger_path,
    storage=pg_config,
)

total_anchors = sum(len(p.section_anchors) for p in ledger.pages)
print(f"Total section anchors across {len(ledger.pages)} pages: {total_anchors}")

if total_anchors == 0:
    print("(No section anchors found — expected when gateway=None or VLM did not emit markers)")
else:
    for page in ledger.pages[:5]:
        for anchor in page.section_anchors:
            print(f"  Page {anchor.page_index}: L{anchor.level} — {anchor.title!r} (offset {anchor.char_offset})")

In [ ]:
from nullvector.domain.tree import TreeBuildManifest

tree_manifest_obj = load_model_artifact(
    TreeBuildManifest,
    result.tree_manifest_path,
    storage=pg_config,
)

show_json("Tree build settings", {
    "hierarchy_chunk_size": tree_manifest_obj.settings.hierarchy_chunk_size,
    "hierarchy_chunking_threshold": tree_manifest_obj.settings.hierarchy_chunking_threshold,
    "committed_node_count": tree_manifest_obj.committed_node_count,
})

# Load committed hierarchy nodes and show source anchors
import json as _json

from nullvector.domain.tree import HierarchyNode
from nullvector.storage import build_document_store

store = build_document_store(pg_config)
raw_nodes = store.read_json_artifact(tree_manifest_obj.committed_hierarchy_path)

# Use model_validate_json to handle strict StrEnum coercion from JSON strings
nodes = tuple(
    HierarchyNode.model_validate_json(_json.dumps(n)) for n in raw_nodes
)

anchored = sum(1 for n in nodes if n.source_anchors)
print(f"\nNodes with source anchors: {anchored}/{len(nodes)}")

for node in nodes[:8]:
    anchors_str = f" | anchors={len(node.source_anchors)}" if node.source_anchors else ""
    print(f"  {"/".join(node.path)} | pages {node.page_span.start_page}-{node.page_span.end_page}{anchors_str}")
    for anchor in node.source_anchors:
        print(f"    anchor: page {anchor.page}, quote={anchor.quote!r}")


## Tree Summarization (LLM-Powered)

When a live gateway is available, `client.build_tree(summarize=True)` re-builds the
hierarchy tree with LLM-generated summaries for each node. This enriches search and QA
with semantic context beyond raw text.

**Note:** The initial tree build (inside `ingest()`) already uses LLM synthesis to produce
the hierarchy. For documents with 15+ pages, this uses a map-reduce strategy — chunking the
document into overlapping windows, analyzing each in parallel, and merging the results. The
`summarize=True` flag adds a *separate* bottom-up summarization pass on top of that.

When `gateway=None`, this step is a no-op — the tree stays deterministic.

In [ ]:
if gateway is not None:
    tree_manifest = client.build_tree(
        result.acquisition_manifest_path,
        summarize=True,
        tree_run_id="gemini-001-summarized",
    )
    show_json("Tree re-built with LLM summaries", {
        "tree_run_id": tree_manifest.tree_run_id,
        "committed_nodes": tree_manifest.committed_node_count,
    })
else:
    print("No gateway — tree remains deterministic (no LLM summaries).")

## Document Description

Generates a document-level summary from the top-level tree nodes. This is reused by
`client.ask()` — when you ask "what is this book about?", NullVector returns the
pre-built description instead of running retrieval.

In [ ]:
description = client.build_description(
    result.acquisition_manifest_path,
    result.tree_manifest_path,
)

show_json("Description built", {
    "document_id": description.document_id,
    "description_run_id": description.description_run_id,
})

## Search

`client.search()` finds relevant sections using BM25-style structural ranking over
the hierarchical tree. Results include page spans for exact citations.

In [ ]:
hits = client.search("sets", ingest_result=result, limit=5)

for i, hit in enumerate(hits, 1):
    unit = hit.unit
    excerpt = (unit.text or "")[:150].replace("\n", " ")
    print(f"[{i}] score={hit.score:.3f} | pages {unit.page_span.start_page}-{unit.page_span.end_page}")
    print(f"    type={unit.unit_type.value} | {excerpt}...")
    print()

## Grounded QA — Three Answer Modes

`client.ask()` classifies the query intent and picks the right strategy:

- **document_summary** — "What is this book about?" → uses the pre-built description, no retrieval
- **focused_lookup** — "What does the Sets section introduce?" → retrieves and synthesizes with page citations
- **low_evidence** — "zebra invoice compliance" → acknowledges the document doesn't contain relevant content

In [ ]:
qa_examples = {
    "document_summary": "What is this book about?",
    "focused_lookup": "What does the Sets section introduce?",
    "low_evidence": "zebra invoice compliance",
}

for label, query in qa_examples.items():
    response = client.ask(query, ingest_result=result)
    print(f"Q: {query}")
    print(f"  mode={response.answer_mode} | strategy={response.answer_strategy}")
    print(f"  answer: {response.answer[:200]}...")
    if response.citations:
        for c in response.citations:
            print(f"  citation: page {c.page_label} — {(c.quote or '')[:80]}")
    print()

## Results Summary

In [ ]:
show_json("Pipeline summary", {
    "document_id": result.document_id,
    "workspace": str(WORKSPACE),
    "gateway_mode": gateway_mode,
    "acquisition_manifest": result.acquisition_manifest_path,
    "tree_manifest": result.tree_manifest_path,
    "retrieval_manifest": result.retrieval_manifest_path,
})

## Notes

- `client.ingest()` combines acquisition, tree build, and retrieval corpus construction into one call
- `client.build_tree(summarize=True)` re-builds the tree with LLM-generated summaries when a gateway is available
- `client.build_description()` is separate because it's optional and can use a live LLM gateway
- `client.search()` and `client.ask()` accept `ingest_result=`, `document_id=`, or `retrieval_manifest_path=`
- All artifacts are persisted in PostgreSQL — omitted client run IDs now generate fresh runs by default
- When `gateway=None` (no LiteLLM), the pipeline uses outline-based fallback — no LLM/VLM calls
- When a live Groq gateway is available, VLM transcription, LLM hierarchy synthesis, summarization, and descriptions are enabled
- Acquisition uses parallel VLM transcription (default: 4 concurrent pages). Override via `DocumentPreset` with custom `AcquisitionSettings(vlm_max_concurrent_pages=N)`
- The VLM emits `SECTION_ANCHOR` markers that are matched to hierarchy nodes as `source_anchors` for provenance
- Documents with 15+ pages automatically use map-reduce hierarchy synthesis (configurable via `TreeSettings`)
- The `HierarchyBuildReport.synthesis_method` field reports `llm`, `llm_map_reduce`, `outline_fallback`, or `fallback_single_node`
- For service-level APIs (tree search, preference search, compaction), import from `nullvector.retrieval` and `nullvector.tree`
- Set `NULLVECTOR_NOTEBOOK_ENABLE_BRAIN=0` to force deterministic-only mode even when LiteLLM is available